### 1. Importing Libraries

In [1]:
import pandas as pd

### 2. Loading Raw Data

In [2]:
df = pd.read_csv(
    "../data/DATA.txt",
    sep=";",
    decimal=",",
    skipinitialspace=True
)

# remove completely empty columns created by trailing delimiters
df = df.dropna(axis=1, how="all")

# rename columns
df.columns = [
    "timestamp",
    "wind_speed",
    "pitch_angle",
    "power_output"
]

### 3. Initial Data Exploration

In [3]:
# assessing the different variables, exploring for anomalies
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250579 entries, 0 to 250578
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   timestamp     250579 non-null  object 
 1   wind_speed    248029 non-null  float64
 2   pitch_angle   248029 non-null  float64
 3   power_output  248029 non-null  float64
dtypes: float64(3), object(1)
memory usage: 7.6+ MB


In [4]:
# investigating why there are null values, and if they are null across all measures (yes)
df[df.isnull().any(axis=1)].head()

,timestamp,wind_speed,pitch_angle,power_output
5372,07/01/2020 07.20.00,NaN,NaN,NaN
16562,25/03/2020 00.20.00,NaN,NaN,NaN
16563,25/03/2020 00.30.00,NaN,NaN,NaN
16564,25/03/2020 00.40.00,NaN,NaN,NaN
16565,25/03/2020 00.50.00,NaN,NaN,NaN


In [5]:
# assessing if missing values occur in blocks, or random points (this case is a mix)
missing = df[df.isnull().any(axis=1)]
missing.head(10)

,timestamp,wind_speed,pitch_angle,power_output
5372,07/01/2020 07.20.00,NaN,NaN,NaN
16562,25/03/2020 00.20.00,NaN,NaN,NaN
16563,25/03/2020 00.30.00,NaN,NaN,NaN
16564,25/03/2020 00.40.00,NaN,NaN,NaN
16565,25/03/2020 00.50.00,NaN,NaN,NaN
16566,25/03/2020 01.00.00,NaN,NaN,NaN
16567,25/03/2020 01.10.00,NaN,NaN,NaN
16568,25/03/2020 01.20.00,NaN,NaN,NaN
16569,25/03/2020 01.30.00,NaN,NaN,NaN
16570,25/03/2020 01.40.00,NaN,NaN,NaN


In [6]:
# finding the % of all data points that are missing values
missing_count = df["wind_speed"].isna().sum()
total_count = len(df)
missing_percentage = (missing_count / total_count) * 100
missing_percentage

np.float64(1.0176431384912543)

In [7]:
df["missing_data"] = df["wind_speed"].isna()
df["missing_data"].value_counts()

missing_data
False    248029
True       2550
Name: count, dtype: int64

### 4. Data Cleaning / Formatting

In [8]:
# assessing differences in formatting of date/time values
df["timestamp"].str.len().value_counts()

timestamp
19    248838
18      1741
Name: count, dtype: int64

In [9]:
# removing whitespace from time values
df["timestamp"] = df["timestamp"].str.strip()

In [10]:
# converted time slots from midnight to now have the numeric time also
date_only = df["timestamp"].str.len() == 10
df.loc[date_only, "timestamp"] = (
    df.loc[date_only, "timestamp"] + " 00:00:00"
)
df["timestamp"].head()

0    01/12/2019 00:00:00
1    01/12/2019 00.10.00
2    01/12/2019 00.20.00
3    01/12/2019 00.30.00
4    01/12/2019 00.40.00
Name: timestamp, dtype: object

In [11]:
# verifying that all time data points are of same format (length) - this worked
df["timestamp"].str.len().value_counts()

timestamp
19    250579
Name: count, dtype: int64

In [ ]:
# stanardising formatting to colons not periods (as required for 'to_datetime'total_count
df["timestamp"] = df["timestamp"].str.replace(".", ":", regex=False)

# converting strings to datetime object
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    dayfirst=True
)

In [ ]:
# comfirming conversion of Dtype for 'timestamp' is now datetime (success)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250579 entries, 0 to 250578
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   timestamp     250579 non-null  datetime64[ns]
 1   wind_speed    248029 non-null  float64       
 2   pitch_angle   248029 non-null  float64       
 3   power_output  248029 non-null  float64       
 4   missing_data  250579 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(3)
memory usage: 7.9 MB


In [ ]:
# sampling data range
df["timestamp"].min(), df["timestamp"].max()

(Timestamp('2019-12-01 00:00:00'), Timestamp('2024-09-05 03:00:00'))

In [ ]:
# sampling frequency check (every 10 min)
df["timestamp"].diff().value_counts().head()

timestamp
0 days 00:10:00    250578
Name: count, dtype: int64